In [1]:
import pandas as pd
import numpy as np
import joblib
import category_encoders as ce
import optuna
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier


class KingaMetricCreditRiskModel:
    def __init__(self):
        self.models = {}
        self.weights = None
        self.target_encoder = None
        self.feature_names = None
        self.best_threshold = 0.5
        self.best_params = {}
        
        self.leaky_features = [
            'Payment_Behaviour',
            'Delay_from_due_date',
            'Num_of_Delayed_Payment'
        ]

    # =========================
    # DATA PREP
    # =========================
    def load_and_preprocess(self, filepath):
        df = pd.read_csv(filepath)

        df = df.drop(columns=[c for c in self.leaky_features if c in df.columns], errors='ignore')

        df = self.add_interaction_features(df)

        df.replace([np.inf, -np.inf], np.nan, inplace=True)
        df.fillna(df.median(numeric_only=True), inplace=True)

        X = df.drop('Default_Flag', axis=1)
        y = df['Default_Flag']

        return X, y

    def add_interaction_features(self, df):
        df = df.copy()

        # Original
        df["Debt_Stress"] = df["normalized_dti"] * df["normalized_utilization"]
        df["Repayment_Stress"] = df["normalized_emi"] * df["normalized_delinquency"]
        df["Liquidity_Index"] = df["normalized_savings"] * df["normalized_emi"]
        df["Credit_Exposure"] = df["Num_Credit_Card"] * df["Credit_Utilization_Ratio"]
        df["Risk_Index"] = (
            df["normalized_dti"] +
            df["normalized_utilization"] +
            df["normalized_delinquency"]
        ) / 3

        # New advanced
        df["Income_Delinq"] = df["Annual_Income"] * df["normalized_delinquency"]
        #df["Age_Util"] = df["Age"] * df["normalized_utilization"]
        df["Loan_DTI"] = df["Num_of_Loan"] * df["normalized_dti"]

        # Bins
        #df["Age_Group"] = pd.cut(df["Age"], bins=[0, 30, 40, 50, 65, 100], labels=['Young', 'Adult', 'Middle', 'Senior', 'Elder']).astype('str')
        df["Income_Q"] = pd.qcut(df["Annual_Income"], 4, labels=['Q1', 'Q2', 'Q3', 'Q4']).astype('str')

        # Poly top features (util, dti, delinq)
        for feat in ['Credit_Utilization_Ratio', 'normalized_dti', 'normalized_delinquency']:
            df[f'{feat}_sq'] = df[feat]**2
            df[f'{feat}_log'] = np.log1p(df[feat])

        return df

    # =========================
    # ENCODING
    # =========================
    def target_encode(self, X, y=None, fit=False):
        cat_cols = [c for c in ["Payment_of_Min_Amount", "Credit_Mix", "Borrower_Tier", "Age_Group", "Income_Q"] if c in X.columns]

        if fit:
            encoder = ce.TargetEncoder(cols=cat_cols, smoothing=10)
            X = encoder.fit_transform(X, y)
            self.target_encoder = encoder
        else:
            if self.target_encoder is None:
                raise ValueError("Target encoder not fitted.")
            X = self.target_encoder.transform(X)

        return X

    # =========================
    # FEATURE SELECTION (CV STABLE)
    # =========================
    def feature_selection(self, X, y):
        from sklearn.feature_selection import mutual_info_classif

        # MI
        mi_scores = mutual_info_classif(X, y, random_state=42)
        mi_scores = pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)

        # Keep top 30 (all after FE)
        top_features = mi_scores.head(30).index
        return X[top_features], top_features.tolist()

    # =========================
    # MODEL TRAINING
    # =========================
    def tune_hyperparams(self, X, y):
        def xgb_objective(trial):
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
                'max_depth': trial.suggest_int('max_depth', 4, 10),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
                'subsample': trial.suggest_float('subsample', 0.7, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
                'reg_alpha': trial.suggest_float('reg_alpha', 0, 10),
                'reg_lambda': trial.suggest_float('reg_lambda', 0, 10),
                'scale_pos_weight': (y==0).sum() / (y==1).sum()
            }
            model = XGBClassifier(**params, tree_method='hist', random_state=42)
            cv = TimeSeriesSplit(n_splits=5)
            scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')
            return scores.mean()

        def lgbm_objective(trial):
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
                'num_leaves': trial.suggest_int('num_leaves', 20, 100),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
                'subsample': trial.suggest_float('subsample', 0.7, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
                'reg_alpha': trial.suggest_float('reg_alpha', 0, 10),
                'reg_lambda': trial.suggest_float('reg_lambda', 0, 10),
            }
            model = LGBMClassifier(**params, force_col_wise=True, random_state=42, verbose=-1)
            cv = TimeSeriesSplit(n_splits=5)
            scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')
            return scores.mean()

        def rf_objective(trial):
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
                'max_depth': trial.suggest_int('max_depth', 4, 12),
                'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
                'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
                'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
                'class_weight': trial.suggest_categorical('class_weight', [None, 'balanced'])
            }
            model = RandomForestClassifier(**params, random_state=42, n_jobs=-1)
            cv = TimeSeriesSplit(n_splits=5)
            scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')
            return scores.mean()

        def cat_objective(trial):
            params = {
                'iterations': trial.suggest_int('iterations', 200, 1000),
                'depth': trial.suggest_int('depth', 4, 10),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
                'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
                'border_count': trial.suggest_int('border_count', 32, 255),
            }
            model = CatBoostClassifier(**params, verbose=0, random_state=42, auto_class_weights='Balanced')
            cv = TimeSeriesSplit(n_splits=5)
            scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')
            return scores.mean()

        print("Tuning RF...")
        rf_study = optuna.create_study(direction='maximize')
        rf_study.optimize(rf_objective, n_trials=50)
        self.best_params['rf'] = rf_study.best_params

        print("Tuning XGB...")
        xgb_study = optuna.create_study(direction='maximize')
        xgb_study.optimize(xgb_objective, n_trials=50)
        self.best_params['xgb'] = xgb_study.best_params

        print("Tuning LGBM...")
        lgbm_study = optuna.create_study(direction='maximize')
        lgbm_study.optimize(lgbm_objective, n_trials=50)
        self.best_params['lgbm'] = lgbm_study.best_params

        print("Tuning Cat...")
        cat_study = optuna.create_study(direction='maximize')
        cat_study.optimize(cat_objective, n_trials=50)
        self.best_params['cat'] = cat_study.best_params

        print("Best params:", self.best_params)

    def train_models(self, X, y):
        self.tune_hyperparams(X, y)

        scale = (y == 0).sum() / (y == 1).sum()

        scale = (y == 0).sum() / (y == 1).sum()

        self.base_models = {
            "rf": RandomForestClassifier(**self.best_params['rf'], random_state=42, n_jobs=-1),
            "xgb": XGBClassifier(**self.best_params['xgb'], tree_method="hist", random_state=42, scale_pos_weight=scale),
            "lgbm": LGBMClassifier(**self.best_params['lgbm'], force_col_wise=True, random_state=42, verbose=-1, class_weight='balanced'),
            "cat": CatBoostClassifier(**self.best_params['cat'], verbose=0, random_state=42, auto_class_weights="Balanced")
        }

        self.stacking_model = StackingClassifier(
            estimators=list(self.base_models.items()),
            final_estimator=LogisticRegression(random_state=42),
            cv=5,
            stack_method="predict_proba"
        )

        self.stacking_model.fit(X, y)

    # =========================
    # LOGIT FUNCTION
    # =========================
    def _logit(self, p):
        p = np.clip(p, 1e-6, 1 - 1e-6)
        return np.log(p / (1 - p))

    # =========================
    # ENSEMBLE WEIGHTS (FIXED)
    # =========================
    # Legacy logit, disabled for stacking
    def optimize_weights(self, X, y):
        print("Using stacking - no weights optimization needed.")

    # =========================
    # THRESHOLD
    # =========================
    def optimize_threshold(self, y_true, y_probs):
        thresholds = np.linspace(0.1, 0.9, 50)
        best_ks, best_t = 0, 0.5

        for t in thresholds:
            preds = (y_probs >= t).astype(int)

            tpr = ((preds == 1) & (y_true == 1)).sum() / (y_true == 1).sum()
            fpr = ((preds == 1) & (y_true == 0)).sum() / (y_true == 0).sum()

            ks = tpr - fpr

            if ks > best_ks:
                best_ks = ks
                best_t = t

        self.best_threshold = best_t

    # =========================
    # TRAIN PIPELINE
    # =========================
    def train(self, filepath):
        print("Loading...")
        X, y = self.load_and_preprocess(filepath)

        X_temp, X_test, y_temp, y_test = train_test_split(
            X, y, test_size=0.2, stratify=y, random_state=42
        )

        # ✅ Encode
        X_temp = self.target_encode(X_temp, y_temp, fit=True)
        X_test_encoded = self.target_encode(X_test, fit=False)

        # ✅ Feature selection
        X_temp, selected = self.feature_selection(X_temp, y_temp)
        X_test_encoded = X_test_encoded[selected]
        self.feature_names = selected

        # ✅ Train models
        self.train_models(X_temp, y_temp)

        # Stacking already trained, skip weights


        test_probs = self.predict_proba(X_test, already_encoded=False)
        auc = roc_auc_score(y_test, test_probs)

        self.optimize_threshold(y_test, test_probs)

        print(f"Final Test AUC: {auc:.4f}")
        print(f"Best Threshold: {self.best_threshold:.3f}")

        return auc

    # =========================
    # PREDICT
    # =========================
    def predict_proba(self, X, already_encoded=False):
        X = self.add_interaction_features(X)

        if not already_encoded:
            X = self.target_encoder.transform(X)

        X = X[self.feature_names]

        return self.stacking_model.predict_proba(X)[:, 1]

    def predict(self, X):
        probs = self.predict_proba(X)
        return (probs >= self.best_threshold).astype(int)

    def save(self, path="pickled_models/kinga_ensemble_0.pkl"):
        joblib.dump(self, path)
        print(f"Saved to {path}")


if __name__ == '__main__':
    model = KingaMetricCreditRiskModel()
    auc = model.train('datasets/kingametric_credit_risk.csv')
    model.save()

c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading...


[I 2026-03-29 11:17:07,675] A new study created in memory with name: no-name-fcb532f6-cf43-4e6f-8750-c82c2a882293


Tuning RF...


[I 2026-03-29 11:17:12,529] Trial 0 finished with value: 0.6255901359124093 and parameters: {'n_estimators': 404, 'max_depth': 11, 'min_samples_split': 11, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'class_weight': None}. Best is trial 0 with value: 0.6255901359124093.
[I 2026-03-29 11:17:17,947] Trial 1 finished with value: 0.6307173141127198 and parameters: {'n_estimators': 494, 'max_depth': 10, 'min_samples_split': 3, 'min_samples_leaf': 8, 'max_features': 'sqrt', 'class_weight': None}. Best is trial 1 with value: 0.6307173141127198.
[I 2026-03-29 11:17:22,516] Trial 2 finished with value: 0.6389564559890202 and parameters: {'n_estimators': 487, 'max_depth': 5, 'min_samples_split': 12, 'min_samples_leaf': 8, 'max_features': 'sqrt', 'class_weight': None}. Best is trial 2 with value: 0.6389564559890202.
[I 2026-03-29 11:17:32,454] Trial 3 finished with value: 0.6283555628822499 and parameters: {'n_estimators': 847, 'max_depth': 12, 'min_samples_split': 16, 'min_samples_leaf': 7, '

...

Tuning LGBM...


[I 2026-03-29 11:26:55,562] Trial 0 finished with value: 0.6009552142368234 and parameters: {'n_estimators': 403, 'num_leaves': 91, 'learning_rate': 0.20197696371214058, 'subsample': 0.9154279260636459, 'colsample_bytree': 0.8031531309191539, 'reg_alpha': 3.2598846095701606, 'reg_lambda': 1.5010591460624245}. Best is trial 0 with value: 0.6009552142368234.
[I 2026-03-29 11:26:56,011] Trial 1 finished with value: 0.6266117411338878 and parameters: {'n_estimators': 857, 'num_leaves': 39, 'learning_rate': 0.29120352096041363, 'subsample': 0.7905019291548946, 'colsample_bytree': 0.7833748617801591, 'reg_alpha': 9.85824399255162, 'reg_lambda': 8.120155041281068}. Best is trial 1 with value: 0.6266117411338878.
[I 2026-03-29 11:26:56,633] Trial 2 finished with value: 0.6137177672417459 and parameters: {'n_estimators': 288, 'num_leaves': 28, 'learning_rate': 0.133736543303455, 'subsample': 0.9389600550924542, 'colsample_bytree': 0.7336204165878518, 'reg_alpha': 4.835808687484045, 'reg_lambda'

Tuning Cat...


[I 2026-03-29 11:28:45,346] Trial 0 finished with value: 0.589221785370589 and parameters: {'iterations': 972, 'depth': 8, 'learning_rate': 0.2969705542310206, 'l2_leaf_reg': 9.240372635289416, 'border_count': 255}. Best is trial 0 with value: 0.589221785370589.
[I 2026-03-29 11:29:34,257] Trial 1 finished with value: 0.6002532137726915 and parameters: {'iterations': 306, 'depth': 10, 'learning_rate': 0.14147190711626415, 'l2_leaf_reg': 4.617484943675756, 'border_count': 246}. Best is trial 1 with value: 0.6002532137726915.
[I 2026-03-29 11:29:36,987] Trial 2 finished with value: 0.5884755960370905 and parameters: {'iterations': 225, 'depth': 6, 'learning_rate': 0.2945273494096097, 'l2_leaf_reg': 3.1185602084035478, 'border_count': 85}. Best is trial 1 with value: 0.6002532137726915.
[I 2026-03-29 11:29:44,431] Trial 3 finished with value: 0.6191213244676446 and parameters: {'iterations': 840, 'depth': 5, 'learning_rate': 0.02348912337104145, 'l2_leaf_reg': 2.020388663722101, 'border_c

Best params: {'rf': {'n_estimators': 460, 'max_depth': 4, 'min_samples_split': 7, 'min_samples_leaf': 8, 'max_features': None, 'class_weight': None}, 'xgb': {'n_estimators': 430, 'max_depth': 4, 'learning_rate': 0.011428470477839327, 'subsample': 0.8940362813326697, 'colsample_bytree': 0.7250627525433337, 'reg_alpha': 1.4913690667729074, 'reg_lambda': 7.463346775560514}, 'lgbm': {'n_estimators': 777, 'num_leaves': 22, 'learning_rate': 0.010112001077434075, 'subsample': 0.7194769204588124, 'colsample_bytree': 0.9615946118649805, 'reg_alpha': 9.055848591044873, 'reg_lambda': 8.991135105776511}, 'cat': {'iterations': 357, 'depth': 4, 'learning_rate': 0.024466915504627908, 'l2_leaf_reg': 1.3432358457038989, 'border_count': 152}}
Final Test AUC: 0.6355
Best Threshold: 0.329
Saved to pickled_models/kinga_ensemble_0.pkl
